# Exporta.co — Pipeline de Datos
**Maestría en Ciencia de Datos | Visualización y Storytelling con Datos**  
Responsable: Camilo José Beltrán Rojas | Apoyo: Juan Camilo Piñeros, Edgar Ivan Morales Rojas

Estructura modular. Cada sección puede ejecutarse independientemente una vez completadas las anteriores.  
Para migrar a Colab: cambiar `BASE_DIR` en la Sección 1 y subir los ZIPs.

---
## Sección 1 — Rutas
Unico bloque que cambia entre ejecución local y Colab.

In [ ]:
from pathlib import Path

# ── MODO DE EJECUCIÓN ────────────────────────────────────────────────────────
# 'colab' Cambiar a 'local' si se ejecuta fuera de Colab
MODO = 'colab'

if MODO == 'local':
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DIR  = Path('/content/drive/MyDrive/exporta_co')
    RAW_DIR    = DRIVE_DIR / 'raw'
    CLEAN_DIR  = DRIVE_DIR / 'clean'

    # Clonar o actualizar el repositorio
    import subprocess
    REPO_URL  = 'https://github.com/CamiloJose90/Exporta_Co.git'
    REPO_DIR  = Path('/content/Exporta_Co')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

    OUTPUT_DIR = REPO_DIR / 'data'

else:
    RAW_DIR    = Path(r'C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto')
    CLEAN_DIR  = RAW_DIR / 'clean'
    OUTPUT_DIR = RAW_DIR / 'json_output'

CLEAN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# BASE_DIR apunta a donde están los ZIPs (usado en S4)
BASE_DIR = RAW_DIR

print(f'RAW_DIR   : {RAW_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto
C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto\json_output


---
## Sección 2 — Imports

In [67]:
import importlib, subprocess, sys
if importlib.util.find_spec('pycountry') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pycountry', '-q'])

import pandas as pd
import numpy as np
import json, zipfile, io, re
from pathlib import Path
from collections import defaultdict

print(f'pandas {pd.__version__} | numpy {np.__version__}')

pandas 2.3.0 | numpy 2.3.5


---
## Sección 3 — Configuración
Centraliza parámetros, mapas de columnas y diccionarios de referencia.  
Si el DANE cambia nombres de columnas en un año nuevo, solo editar aquí.

In [68]:
# Rango de años a procesar
AÑOS = list(range(2011, 2026))

# Mapeo columnas EXPO: nombre real en CSV -> nombre interno del pipeline
COLUMNAS_EXPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'cod_pai4': 'pais',          # ya viene en ISO alfa-3 (USA, DEU...)
    'posar':    'subpartida',    # posición arancelaria HS
    'fobdol':   'fob_usd',
    'pnk':      'kg_neto',
}

# Mapeo columnas IMPO
COLUMNAS_IMPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'paisgen':  'pais',          # código numérico DIAN
    'naban':    'subpartida',
    'vacid':    'cif_usd',       # valor CIF dólares
    'pnk':      'kg_neto',
}

# Capítulos HS -> sector OMC (para derivar sector desde posición arancelaria)
# 01-24: Agropecuarios | 25-27: Combustibles | 28-97: Manufacturas
_SECTOR_BREAKS = [0, 24, 27, 97]
_SECTOR_LABELS = [
    'Agropecuarios, alimentos y bebidas',
    'Combustibles e industrias extractivas',
    'Manufacturas',
]

# Mes en español -> número (para extraer mes desde nombre de archivo IMPO)
MESES_ES = {
    'enero':1, 'febrero':2, 'marzo':3, 'abril':4,
    'mayo':5, 'junio':6, 'julio':7, 'agosto':8,
    'septiembre':9, 'octubre':10, 'noviembre':11, 'diciembre':12,
}

print('Configuracion cargada')
print(f'Anos: {AÑOS[0]}-{AÑOS[-1]}')

Configuracion cargada
Anos: 2011-2025


---
## Sección 4 — Descubrimiento de archivos
Escanea BASE_DIR y agrupa ZIPs por tipo y año.  
Maneja años partidos en sufijos _1 y _2 automáticamente.

In [69]:
PATRON = re.compile(r'^(expo|impo)_(\d{4})(?:_(\d))?\.zip$', re.IGNORECASE)
archivos = {'expo': defaultdict(list), 'impo': defaultdict(list)}

for f in sorted(BASE_DIR.glob('*.zip')):
    m = PATRON.match(f.name)
    if m:
        tipo = m.group(1).lower()
        año  = int(m.group(2))
        archivos[tipo][año].append(f)

for tipo in ('expo', 'impo'):
    años_rango   = sorted(a for a in archivos[tipo] if a in AÑOS)
    años_fuera   = sorted(a for a in archivos[tipo] if a not in AÑOS)
    años_faltant = [a for a in AÑOS if a not in archivos[tipo]]
    print(f'\n{tipo.upper()} -- {len(años_rango)} año(s) en rango')
    for año in años_rango:
        partes = archivos[tipo][año]
        tag = f'[{len(partes)} partes]' if len(partes) > 1 else ''
        for p in partes:
            print(f'  {p.name} {tag}')
    if años_faltant:
        print(f'  Faltantes en rango: {años_faltant}')
    if años_fuera:
        print(f'  Fuera de rango (ignorados): {años_fuera}')


EXPO -- 15 año(s) en rango
  Expo_2011.zip 
  Expo_2012.zip 
  Expo_2013.zip 
  Expo_2014.zip 
  Expo_2015.zip 
  Expo_2016.zip 
  Expo_2017.zip 
  Expo_2018.zip 
  Expo_2019.zip 
  Expo_2020.zip 
  Expo_2021.zip 
  Expo_2022.zip 
  Expo_2023.zip 
  Expo_2024.zip 
  Expo_2025.zip 

IMPO -- 15 año(s) en rango
  Impo_2011.zip 
  Impo_2012.zip 
  Impo_2013.zip 
  Impo_2014.zip 
  Impo_2015.zip 
  Impo_2016.zip 
  Impo_2017.zip 
  Impo_2018.zip 
  Impo_2019.zip 
  Impo_2020.zip 
  Impo_2021_1.zip [2 partes]
  Impo_2021_2.zip [2 partes]
  Impo_2022_1.zip [2 partes]
  Impo_2022_2.zip [2 partes]
  Impo_2023.zip 
  Impo_2024.zip 
  Impo_2025_1.zip [2 partes]
  Impo_2025_2.zip [2 partes]


---
## Sección 5 — Funciones de ingesta
Funciones separadas para EXPO e IMPO dado que difieren en:
- Formato de fecha: EXPO usa YYMM, IMPO usa YYYY (mes se extrae del nombre del archivo)
- Identificador de país: EXPO trae ISO alfa-3 directo, IMPO trae código numérico DIAN

In [70]:
def normalizar_columnas(df, mapa):
    rename = {}
    for col in df.columns:
        clave = col.strip().lower().replace(' ', '_')
        if clave in mapa:
            rename[col] = mapa[clave]
        elif col.strip() in mapa:
            rename[col] = mapa[col.strip()]
    return df.rename(columns=rename)


def limpiar_valor_monetario(serie):
    """Maneja formato europeo: punto=miles, coma=decimal (ej: 4.885,70 -> 4885.70)."""
    return (
        serie.astype(str)
        .str.strip()
        .str.replace('.', '', regex=False)   # eliminar separador de miles
        .str.replace(',', '.', regex=False)   # coma decimal -> punto
        .pipe(pd.to_numeric, errors='coerce')
    )


def derivar_sector(serie_subpartida):
    """Deriva sector OMC desde los 2 primeros dígitos de la posición arancelaria (capítulo HS)."""
    capitulo = pd.to_numeric(serie_subpartida.astype(str).str[:2], errors='coerce')
    return (
        pd.cut(capitulo, bins=_SECTOR_BREAKS, labels=_SECTOR_LABELS)
        .astype(str)
        .replace('nan', 'Otros sectores')
    )


def mes_desde_nombre(nombre_archivo):
    """Extrae número de mes desde nombre de archivo (ej: 'Agosto 2024/Agosto.csv' -> 8)."""
    stem = Path(nombre_archivo).stem.lower().strip()
    return MESES_ES.get(stem, None)


def leer_csv_en_memoria(data_bytes):
    """Lee CSV desde bytes probando encodings y separadores comunes del DANE."""
    for encoding in ('latin-1', 'utf-8', 'utf-8-sig'):
        for sep in (',', ';', '\t'):
            try:
                df = pd.read_csv(
                    io.BytesIO(data_bytes),
                    sep=sep,
                    encoding=encoding,
                    low_memory=False,
                    on_bad_lines='skip',
                )
                if len(df.columns) > 2:
                    return df.dropna(how='all')
            except Exception:
                continue
    return None


def leer_zip_expo(ruta_zip):
    """
    Estructura: ZIP anual -> ZIPs mensuales -> CSV
    Año y mes se derivan de FECH (formato YYMM).
    """
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom_zip in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_en_memoria(zf_mes.read(csv_path))
                            if df is not None:
                                frames.append(normalizar_columnas(df, COLUMNAS_EXPO))
                except Exception as e:
                    print(f'    Advertencia {nom_zip}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def leer_zip_impo(ruta_zip):
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            contenidos = zf.namelist()
            zips_mes   = sorted(f for f in contenidos if f.lower().endswith('.zip'))
            csvs_dir   = sorted(f for f in contenidos if f.lower().endswith('.csv'))

            # Caso A: ZIPs mensuales internos (2011-2023, 2025)
            if zips_mes:
                for nom_zip in zips_mes:
                    try:
                        with zipfile.ZipFile(io.BytesIO(zf.read(nom_zip)), 'r') as zf_mes:
                            for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                                df = leer_csv_en_memoria(zf_mes.read(csv_path))
                                if df is not None:
                                    df = normalizar_columnas(df, COLUMNAS_IMPO)
                                    mes = mes_desde_nombre(csv_path)
                                    if mes:
                                        df['mes'] = mes
                                    frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {nom_zip}: {e}')

            # Caso B: CSVs directos en carpetas (2024)
            elif csvs_dir:
                for csv_path in csvs_dir:
                    try:
                        df = leer_csv_en_memoria(zf.read(csv_path))
                        if df is not None:
                            df = normalizar_columnas(df, COLUMNAS_IMPO)
                            mes = mes_desde_nombre(csv_path)
                            if mes:
                                df['mes'] = mes
                            frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {csv_path}: {e}')

            else:
                print(f'    Sin CSVs ni ZIPs reconocidos en {ruta_zip.name}')

    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


print('Funciones listas')

Funciones listas


---
## Sección 6 — Ingesta
Para exploración, limitar AÑOS en la Sección 3 a un solo año antes de correr todo.

In [71]:
def _ingestar(tipo, fn_leer):
    frames = []
    for año in AÑOS:
        partes = archivos[tipo].get(año, [])
        if not partes:
            print(f'  {tipo.upper()} {año}: no encontrado, omitido')
            continue
        frames_año = []
        for ruta in sorted(partes):
            frames_año.extend(fn_leer(ruta))
        if frames_año:
            df = pd.concat(frames_año, ignore_index=True)
            df['año'] = año
            frames.append(df)
            print(f'  {tipo.upper()} {año}: {len(df):,} filas')
        else:
            print(f'  {tipo.upper()} {año}: sin datos legibles')
    return pd.concat(frames, ignore_index=True) if frames else None


df_expo = _ingestar('expo', leer_zip_expo)
df_impo = _ingestar('impo', leer_zip_impo)

  EXPO 2011: 411,377 filas
  EXPO 2012: 417,899 filas
  EXPO 2013: 415,723 filas
  EXPO 2014: 415,693 filas
  EXPO 2015: 423,357 filas
  EXPO 2016: 443,192 filas
  EXPO 2017: 467,445 filas
  EXPO 2018: 474,460 filas
  EXPO 2019: 492,610 filas
  EXPO 2020: 440,317 filas
  EXPO 2021: 498,945 filas
  EXPO 2022: 495,318 filas
  EXPO 2023: 459,332 filas
  EXPO 2024: 480,072 filas
  EXPO 2025: 502,206 filas
  IMPO 2011: 2,661,426 filas
  IMPO 2012: 2,906,378 filas
  IMPO 2013: 3,017,424 filas
  IMPO 2014: 3,134,920 filas
  IMPO 2015: 2,978,480 filas
  IMPO 2016: 2,920,735 filas
  IMPO 2017: 3,126,595 filas
  IMPO 2018: 3,321,731 filas
  IMPO 2019: 3,504,366 filas
  IMPO 2020: 2,861,743 filas
  IMPO 2021: 3,278,600 filas
  IMPO 2022: 3,855,326 filas
  IMPO 2023: 3,361,397 filas
  IMPO 2024: 3,543,882 filas
  IMPO 2025: 3,821,128 filas


---
## Sección 7 — Diagnostico de columnas
Verificar antes de limpiar. Si hay columnas faltantes, ajustar los mapas en la Sección 3
y re-ejecutar desde la Sección 6.

In [72]:
COLS_EXPO = ['fech', 'pais', 'subpartida', 'fob_usd', 'kg_neto', 'mes']
COLS_IMPO = ['fech', 'pais', 'subpartida', 'cif_usd', 'kg_neto', 'mes']

def diagnostico(df, nombre, cols):
    if df is None:
        print(f'{nombre}: DataFrame vacio'); return
    print(f'\n{nombre}')
    print(f'  Filas    : {len(df):,}')
    print(f'  Anos     : {sorted(df["año"].dropna().unique().astype(int).tolist())}')
    print(f'  Columnas : {list(df.columns)}')
    for c in cols:
        estado = 'OK' if c in df.columns else 'FALTA'
        print(f'  [{estado}] {c}')

diagnostico(df_expo, 'EXPORTACIONES', COLS_EXPO)
diagnostico(df_impo, 'IMPORTACIONES', COLS_IMPO)


EXPORTACIONES
  Filas    : 6,837,946
  Anos     : [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Columnas : ['fech', 'ADUA', 'PAIS', 'pais', 'COD_SAL1', 'COD_SAL', 'DPTO2', 'VIA', 'BANDERA', 'REGIM', 'MODAD', 'FINALID', 'CER_ORI1', 'SISESP', 'subpartida', 'DPTO1', 'UNID', 'CODUNI2', 'CANTI', 'PBK', 'kg_neto', 'fob_usd', 'FOBPES', 'AGRENA', 'FLETES', 'SEGURO', 'OTROSG', 'NIT', 'RAZ_SIAL', 'año', ' PBK ', ' FOBPES ', ' AGRENA ', ' FLETES ', ' SEGURO ', ' OTROSG ', ' CANTI ', 'Unnamed: 27']
  [OK] fech
  [OK] pais
  [OK] subpartida
  [OK] fob_usd
  [OK] kg_neto
  [FALTA] mes

IMPORTACIONES
  Filas    : 48,294,131
  Anos     : [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Columnas : ['fech', 'ADUA', 'pais', 'PAISPRO', 'PAISCOM', 'DEPTODES', 'VIATRANS', 'BANDERA', 'REGIMEN', 'ACUERDO', 'PBK', 'kg_neto', 'CANU', 'CODA', 'subpartida', 'VAFODO', 'FLETE', 'cif_usd', 'VACIP', 'IMP1', 'OTDER', 'CLASE', '

---
## Sección 8 — Limpieza
Ejecutar solo cuando el diagnostico muestre OK en todas las columnas requeridas.

In [73]:
# EXPORTACIONES
print('Limpiando EXPORTACIONES...')
n0 = len(df_expo)

fech = pd.to_numeric(df_expo['fech'], errors='coerce')
df_expo['año'] = (2000 + fech // 100).astype('Int64')
df_expo['mes'] = (fech % 100).astype('Int64')

df_expo['iso3']    = df_expo['pais'].str.strip().str.upper()
df_expo['fob_usd'] = limpiar_valor_monetario(df_expo['fob_usd'])
df_expo['sector']  = derivar_sector(df_expo['subpartida'])

df_expo = df_expo[
    df_expo['fob_usd'].notna() & (df_expo['fob_usd'] > 0) &
    df_expo['iso3'].notna() & (df_expo['iso3'] != '') &
    df_expo['año'].between(AÑOS[0], AÑOS[-1]) &
    df_expo['mes'].between(1, 12)
]
print(f'  Eliminadas: {n0 - len(df_expo):,} | Restantes: {len(df_expo):,}')
print(df_expo[['año','mes','iso3','fob_usd','sector']].head(3))

Limpiando EXPORTACIONES...
  Eliminadas: 2,080 | Restantes: 6,835,866
    año  mes iso3  fob_usd        sector
0  2011    4  USA     92.0  Manufacturas
1  2011    4  DEU   2565.0  Manufacturas
2  2011    4  DEU   5640.0  Manufacturas


In [74]:
# IMPORTACIONES
print('Limpiando IMPORTACIONES...')
n0 = len(df_impo)

# FECH
fech = pd.to_numeric(df_impo['fech'], errors='coerce')
df_impo['año'] = (2000 + fech // 100).astype('Int64')
df_impo['mes'] = (fech % 100).astype('Int64')

# ISO-3 desde código numérico ISO
_cache_iso3_impo = {}
def codigo_numerico_a_iso3(codigo):
    if pd.isna(codigo):
        return None
    key = str(int(codigo)).zfill(3)
    if key not in _cache_iso3_impo:
        try:
            _cache_iso3_impo[key] = pycountry.countries.get(numeric=key).alpha_3
        except Exception:
            _cache_iso3_impo[key] = None
    return _cache_iso3_impo[key]

df_impo['iso3'] = pd.to_numeric(df_impo['pais'], errors='coerce').map(codigo_numerico_a_iso3)

# CIF: coma es separador decimal, no de miles
df_impo['cif_usd'] = (
    df_impo['cif_usd'].astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

df_impo = df_impo[
    df_impo['cif_usd'].notna() & (df_impo['cif_usd'] > 0) &
    df_impo['iso3'].notna() &
    df_impo['año'].between(AÑOS[0], AÑOS[-1]) &
    df_impo['mes'].between(1, 12)
]
print(f'  Eliminadas: {n0 - len(df_impo):,} | Restantes: {len(df_impo):,}')
print(df_impo[['año','mes','iso3','cif_usd']].head(3))

Limpiando IMPORTACIONES...
  Eliminadas: 42,381,815 | Restantes: 5,912,316
       año  mes iso3   cif_usd
8028  2011    4  BWA   3352.92
8029  2011    4  BWA   4262.23
8030  2011    4  BWA  60497.90


---
## Sección 9 — Cache en Parquet
Guarda los DataFrames limpios por año en Drive para evitar reprocesar los ZIPs.
Para ejecuciones siguientes, comentar S6 y S8 y descomentar el bloque de carga rápida.

In [ ]:
COLS_EXPO_CLEAN = ['año', 'mes', 'iso3', 'fob_usd', 'sector', 'subpartida', 'kg_neto']
COLS_IMPO_CLEAN = ['año', 'mes', 'iso3', 'cif_usd', 'subpartida', 'kg_neto']

for año in AÑOS:
    df_e = df_expo[df_expo['año'] == año][COLS_EXPO_CLEAN]
    if len(df_e):
        df_e.to_parquet(CLEAN_DIR / f'expo_{año}.parquet', index=False)

    df_i = df_impo[df_impo['año'] == año][COLS_IMPO_CLEAN]
    if len(df_i):
        df_i.to_parquet(CLEAN_DIR / f'impo_{año}.parquet', index=False)

print(f'Parquet guardados en {CLEAN_DIR}')

## Sección 10 — Carga rápida desde Parquet
Descomentar y ejecutar directamente desde aquí cuando los parquet ya existan en Drive.
Requiere haber ejecutado S1, S2 y S3 previamente.

In [ ]:
# df_expo = pd.concat(
#     [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('expo_*.parquet'))],
#     ignore_index=True
# )
# df_impo = pd.concat(
#     [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('impo_*.parquet'))],
#     ignore_index=True
# )
# print(f'Expo: {len(df_expo):,} filas | Impo: {len(df_impo):,} filas')
print('Celda en modo comentado. Descomentar para carga rapida.')


## Sección 11 — Exportación de JSON para D3.js
Genera los 3 archivos JSON que consume el sitio web.
Output: OUTPUT_DIR/data/ (local o repositorio clonado en Colab).

In [ ]:
# VIZ 1 — Mapa coroplético: FOB por país destino y año
viz1 = (
    df_expo
    .groupby(['año', 'iso3'], as_index=False)['fob_usd']
    .sum()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
    .drop(columns='fob_usd')
    .sort_values(['año', 'fob_musd'], ascending=[True, False])
)
(OUTPUT_DIR / 'viz1_mapa_destinos.json').write_text(
    json.dumps(viz1.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz1: {len(viz1):,} registros | {viz1["iso3"].nunique()} paises')

In [ ]:
# VIZ 2 — Balanza comercial mensual
expo_m = df_expo.groupby(['año','mes'])['fob_usd'].sum().reset_index(name='expo_usd')
impo_m = df_impo.groupby(['año','mes'])['cif_usd'].sum().reset_index(name='impo_usd')
viz2 = (
    pd.merge(expo_m, impo_m, on=['año','mes'], how='outer')
    .fillna(0)
    .assign(
        periodo            = lambda d: d['año'].astype(str) + '-' + d['mes'].astype(str).str.zfill(2),
        exportaciones_musd = lambda d: (d['expo_usd'] / 1e6).round(2),
        importaciones_musd = lambda d: (d['impo_usd'] / 1e6).round(2),
        balanza_musd       = lambda d: ((d['expo_usd'] - d['impo_usd']) / 1e6).round(2),
    )
    .sort_values('periodo')
    [['periodo','año','mes','exportaciones_musd','importaciones_musd','balanza_musd']]
)
(OUTPUT_DIR / 'viz2_balanza_temporal.json').write_text(
    json.dumps(viz2.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz2: {len(viz2):,} periodos ({viz2["periodo"].min()} -> {viz2["periodo"].max()})')

In [ ]:
# VIZ 3 — Treemap sectores OMC
sector_anual = (
    df_expo
    .groupby(['año','sector'])['fob_usd']
    .sum()
    .reset_index()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
)
total_año = sector_anual.groupby('año')['fob_musd'].transform('sum')
sector_anual['participacion_pct'] = (sector_anual['fob_musd'] / total_año * 100).round(1)

años_disp = sorted(sector_anual['año'].unique().tolist())
viz3 = {
    'años_disponibles': años_disp,
    'datos_por_año': {
        str(año): {
            'name': f'Exportaciones {año}',
            'children': [
                {
                    'name': r['sector'],
                    'value': r['fob_musd'],
                    'participacion_pct': r['participacion_pct'],
                }
                for _, r in g.sort_values('fob_musd', ascending=False).iterrows()
            ]
        }
        for año, g in sector_anual.groupby('año')
    }
}
(OUTPUT_DIR / 'viz3_treemap_sectores.json').write_text(
    json.dumps(viz3, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz3: {len(años_disp)} años')

## Sección 12 — Push a GitHub
Solo ejecutar en Colab. Requiere token de acceso configurado.
Instrucciones de token: Settings -> Developer settings -> Personal access tokens -> Fine-grained.

In [ ]:
if MODO == 'colab':
    GITHUB_TOKEN = ''  # pegar token aquí, no subir al repo
    REPO_URL_AUTH = f'https://{GITHUB_TOKEN}@github.com/CamiloJose90/Exporta_Co.git'

    cmds = [
        ['git', '-C', str(REPO_DIR), 'config', 'user.email', 'pipeline@exporta.co'],
        ['git', '-C', str(REPO_DIR), 'config', 'user.name', 'Exporta Pipeline'],
        ['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL_AUTH],
        ['git', '-C', str(REPO_DIR), 'add', 'data/'],
        ['git', '-C', str(REPO_DIR), 'commit', '-m', 'data: actualizar JSONs desde pipeline'],
        ['git', '-C', str(REPO_DIR), 'push'],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f'Error en: {" ".join(cmd[3:])}\n{result.stderr}')
            break
    else:
        print('JSONs pusheados a GitHub correctamente')
else:
    print('Push omitido en modo local. Copiar JSONs de OUTPUT_DIR al repo manualmente.')